In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
df1 = pd.read_csv('loan_approval_data.csv')

In [ ]:
df.head()
df.info()
# df.isnull().sum()
# df.describe()

## Data Cleaning 
# Handle Missing Values

In [ ]:
# numerical missing values are replaced generally by mean value
# for categorical data, we generally replace the missing values by modal values. 

categorical_cols = df1.select_dtypes(include = ['str']).columns
numerical_cols = df1.select_dtypes(include = ['number']).columns
# select_dtypes is used to select cols with specific types of data.


In [ ]:
categorical_cols

In [ ]:
numerical_cols

In [ ]:
# to fill the values, sklearn has a function SimpleImputer.
from sklearn.impute import SimpleImputer

num_imp = SimpleImputer(strategy = "mean")
df1[numerical_cols] = num_imp.fit_transform(df1[numerical_cols])

In [ ]:
cat_imp = SimpleImputer(strategy = "most_frequent")
df1[categorical_cols] = cat_imp.fit_transform(df1[categorical_cols])

# EDA - Exploratory Data Analysis

In [ ]:
# How balanced our classes are

classes_count = df1["Loan_Approved"].value_counts()
plt.pie(classes_count, labels = ["No", "Yes"], autopct = '%1.1f%%')
plt.title("Is Loan approved or not?")

In [ ]:
# analyze categories 
# gender_cnt = df["Gender"].value_counts()
# ax = sns.barplot(gender_cnt)
# ax.bar_label(ax.containers[0])

edctn_cnt = df1["Education_Level"].value_counts()
ax = sns.barplot(edctn_cnt)
ax.bar_label(ax.containers[0])

In [ ]:
# analyze income

sns.histplot(
    data = df1,
    x = "Applicant_Income",
    bins = 20
)

In [ ]:
sns.histplot(
    data = df1,
    x = "Coapplicant_Income",
    bins = 20
)

In [ ]:
# outliers - box plots

sns.boxplot(
    data = df1,
    x = "Loan_Approved",
    y = "Applicant_Income"
    
)

In [ ]:
fig, axes =  plt.subplots(3,2)

sns.boxplot(
    ax = axes[0,0],
    data = df1,
    x = "Loan_Approved",
    y = "Applicant_Income"
)
sns.boxplot(
    ax = axes[0,1],
    data = df1,
    x = "Loan_Approved",
    y = "Credit_Score"
)
sns.boxplot(
    ax = axes[1,0],
    data = df1,
    x = "Loan_Approved",
    y = "DTI_Ratio"
)
sns.boxplot(
    ax = axes[1,1],
    data = df1,
    x = "Loan_Approved",
    y = "Savings"
)
sns.boxplot(
    ax = axes[2,0],
    data = df1,
    x = "Loan_Approved",
    y = "Age"
)
sns.boxplot(
    ax = axes[2,1],
    data = df1,
    x = "Loan_Approved",
    y = "Loan_Amount"
)

plt.tight_layout()

In [ ]:
# Credit score with loan approval

sns.histplot(
    data = df1,
    x = "Credit_Score",
    hue = "Loan_Approved",
    bins = 20,
    multiple = "dodge"
)

In [ ]:
sns.histplot(
    data = df1,
    x = "Applicant_Income",
    hue = "Loan_Approved",
    bins = 20,
    multiple = "dodge"
)

In [ ]:
# Remove the Applicant Id
# it contributes nothing to training, as loan approval doesn't depend on the applicant id1

df1 = df1.drop("Applicant_ID", axis = 1)

# Encoding


In [ ]:
# binary and One Hot Encoding
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

le = LabelEncoder()
df1['Education_Level'] = le.fit_transform(df1["Education_Level"])
df1['Loan_Approved']= le.fit_transform(df1["Loan_Approved"])

In [ ]:
df1.info()

In [ ]:
# one hot encoding
cols = ["Employment_Status","Marital_Status","Loan_Purpose","Property_Area","Gender","Employer_Category"]

ohe = OneHotEncoder(drop = "first", sparse_output = False, handle_unknown = "ignore")

encoded = ohe.fit_transform(df1[cols])

encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cols), index=df1.index)

df1 = pd.concat([df1.drop(columns = cols), encoded_df], axis = 1)

# Correlation Heatmap


In [ ]:
# checks for linear relationships between features

num_cols = df1.select_dtypes(include = "number")
corr_matrix = num_cols.corr()

plt.figure(figsize = (20,10))
sns.heatmap(
    corr_matrix,
    annot = True,
    fmt = ".2f",
    cmap = "coolwarm"
)

In [ ]:
num_cols.corr()["Loan_Approved"].sort_values(ascending = False)

# Train-Test Split + Feature Scaling

In [ ]:
X = df1.drop("Loan_Approved", axis=1)
y = df1["Loan_Approved"]

In [ ]:
y.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
X_test.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train & Evaluate Models
## LogisticRegression, KNN, NaiveBayes

In [ ]:
# Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)

y_pred = log_model.predict(X_test_scaled)

# Evaluation( precision score as we have to minimize the false positives)
print("Logistic Regression Model")
print("Precision : ", precision_score(y_test, y_pred))
print("Recall : ", recall_score(y_test, y_pred))
print("F1 score : ", f1_score(y_test, y_pred))
print("Accuracy : ", accuracy_score(y_test, y_pred))
print("Confusion Matrix : ", confusion_matrix(y_test, y_pred))

In [ ]:
# KNN

from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors = 5)
knn_model.fit(X_train_scaled, y_train)

y_pred = knn_model.predict(X_test_scaled)

# Evaluation( precision score as we have to minimize the false positives)
print("KNN Model")
print("Precision : ", precision_score(y_test, y_pred))
print("Recall : ", recall_score(y_test, y_pred))
print("F1 score : ", f1_score(y_test, y_pred))
print("Accuracy : ", accuracy_score(y_test, y_pred))
print("Confusion Matrix : ", confusion_matrix(y_test, y_pred))

In [ ]:
# Naive Bayes

from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

y_pred = nb_model.predict(X_test_scaled)

# Evaluation( precision score as we have to minimize the false positives)
print("Naive Bayes Model")
print("Precision : ", precision_score(y_test, y_pred))
print("Recall : ", recall_score(y_test, y_pred))
print("F1 score : ", f1_score(y_test, y_pred))
print("Accuracy : ", accuracy_score(y_test, y_pred))
print("Confusion Matrix : ", confusion_matrix(y_test, y_pred))

# Best Model is on the basis of precision -> Naive Bayes

## Feature Engineering

In [ ]:
# ADD of Transform the features

df1["DTI_Ratio_sq"] = df1["DTI_Ratio"] ** 2
df1["Credit_Score_sq"] = df1["Credit_Score"] ** 2

# Taking log of the features which are skewed( just for an example)
df1["Applicant_Income_log"] = np.log1p(df1["Applicant_Income"])

X = df1.drop(columns = ["Loan_Approved", "Credit_Score", "DTI_Ratio", "Applicant_Income"])
y = df1["Loan_Approved"]


# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
X_train.head()

In [ ]:
# Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)

y_pred = log_model.predict(X_test_scaled)

# Evaluation( precision score as we have to minimize the false positives)
print("Logistic Regression Model")
print("Precision : ", precision_score(y_test, y_pred))
print("Recall : ", recall_score(y_test, y_pred))
print("F1 score : ", f1_score(y_test, y_pred))
print("Accuracy : ", accuracy_score(y_test, y_pred))
print("Confusion Matrix : ", confusion_matrix(y_test, y_pred))

In [ ]:
# KNN

from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors = 5)
knn_model.fit(X_train_scaled, y_train)

y_pred = knn_model.predict(X_test_scaled)

# Evaluation( precision score as we have to minimize the false positives)
print("KNN Model")
print("Precision : ", precision_score(y_test, y_pred))
print("Recall : ", recall_score(y_test, y_pred))
print("F1 score : ", f1_score(y_test, y_pred))
print("Accuracy : ", accuracy_score(y_test, y_pred))
print("Confusion Matrix : ", confusion_matrix(y_test, y_pred))

In [ ]:
# Naive Bayes

from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

y_pred = nb_model.predict(X_test_scaled)

# Evaluation( precision score as we have to minimize the false positives)
print("Naive Bayes Model")
print("Precision : ", precision_score(y_test, y_pred))
print("Recall : ", recall_score(y_test, y_pred))
print("F1 score : ", f1_score(y_test, y_pred))
print("Accuracy : ", accuracy_score(y_test, y_pred))
print("Confusion Matrix : ", confusion_matrix(y_test, y_pred))